# Playbook 0 · Acquisition

**Stage:** get NP3-565-CD (forecast vintages) and NP6-345-CD (actuals) onto disk,
preserving when each became available.


> **Playbook, not walkthrough.** [`exploration.ipynb`](exploration.ipynb) is the
> narrative for a reviewer: what was built and why. These five are operational —
> one per assignment stage, each answering *what does this stage guarantee* and
> *what would it take to run it in production*. They overlap deliberately on
> evidence and not at all on purpose.


## What this stage must guarantee

1. **Every file is immutable once written.** A vintage is evidence. If it can be
   overwritten, every downstream cutoff claim becomes unfalsifiable.
2. **Publication time is preserved exactly.** It is the only thing that makes
   point-in-time selection possible, and it is not recoverable later.
3. **Absence is distinguishable from failure.** "ERCOT never published it" and
   "we failed to fetch it" demand opposite responses.

## The two paths, and why there are two

| Path | Credentials | Reach | Role |
| --- | --- | --- | --- |
| `acquire` | none | last ~7 days of the MIS listing | the live tail; runs in tests |
| `backfill` | subscription key **and** bearer token | the authenticated archive | history |

ERCOT's public MIS listing retains roughly seven days of NP3-565. That single
fact drives the production design below more than anything else.

In [1]:
from __future__ import annotations

import datetime as dt
import tempfile
import textwrap
from pathlib import Path

from forecast_spine import coverage, fixtures, gates, normalize, pipeline, seasonal_naive

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO / "data" / "raw"
SQL = REPO / "sql" / "asof_join.sql"

# The assignment window. Actuals for operating day D publish on D+1, so the
# processing date is one day past the last target day.
PROCESSING_DATE = dt.date(2026, 3, 24)
WINDOW_START, WINDOW_END = dt.date(2026, 2, 22), dt.date(2026, 3, 23)
PUBLICATION_START = dt.date(2026, 2, 21)

LIVE = any(RAW.glob("load_forecast/*_csv.zip"))
raw_root = RAW if LIVE else fixtures.build("pass", Path(tempfile.mkdtemp()))
if not LIVE:
    print("LIVE VINTAGES NOT FOUND -- using synthetic fixtures.\n"
          "Structure is preserved; scale and revision behaviour are not.\n")


def build():
    """Build a throwaway warehouse. Never touches data/warehouse/.

    A scratch database keeps this notebook runnable while something else holds
    the committed one -- DuckDB is single-writer, and a SQL client with an open
    connection is enough to block it.
    """
    if LIVE:
        context = pipeline.build_context(
            PROCESSING_DATE, raw_root, window_start=WINDOW_START, window_end=WINDOW_END
        )
    else:
        context = pipeline.build_context(
            fixtures.processing_date_for("pass"), raw_root, window_days=1
        )
    con = pipeline.connect(Path(tempfile.mkdtemp()) / "playbook.duckdb")
    pipeline.load(con, context)
    pipeline.build_evaluation_dataset(con, context, SQL)
    return con, context

## Run it — what is on disk, measured against the calendar

In [2]:
for report_key in ("load_forecast", "actual_load"):
    cov = coverage.measure(report_key, PUBLICATION_START, WINDOW_END, raw_root)
    pct = 100 * cov.held / cov.expected if cov.expected else 0.0
    print(f"{report_key:14s} {cov.held}/{cov.expected} expected publications ({pct:.1f}%)")
    for day in [d for d in cov.days if d.held < d.expected]:
        print(f"               {day.day}  held {day.held}/{day.expected}")
    print()

print("The denominator comes from the calendar, not from 24. `coverage` asks how")
print("many hours the local operating day has, so it never reports the hour that")
print("spring-forward deletes as a missing publication.")

load_forecast  738/743 expected publications (99.3%)
               2026-03-06  held 19/24

actual_load    31/31 expected publications (100.0%)

The denominator comes from the calendar, not from 24. `coverage` asks how
many hours the local operating day has, so it never reports the hour that
spring-forward deletes as a missing publication.


### What a gap costs

A missing publication is **not** the same as a missing forecast. The as-of rule
takes the newest vintage published at or before `T − 24h`, so a gap only matters
when it removes the *last eligible* vintage for some target hour, or leaves one
old enough to trip the freshness check.

For this window that distinction is not academic — it is the difference between
five absent files and sixteen unusable zone-hours.

In [3]:
con, context = build()

stale = con.execute(
    """
    SELECT operating_date, hour_ending,
           round(max(ercot_vintage_age_hours), 2) AS worst_age_hours,
           count(*) AS zone_hours
    FROM evaluation_dataset
    WHERE ercot_vintage_age_hours > 2.0
    GROUP BY 1, 2 ORDER BY 1, 2
    """
).df()

print("Five publications are absent on 2026-03-06. One day later:\n")
print(stale.to_string(index=False))
print()
print("NP3-565 publishes hourly, so the newest vintage publishable by a cutoff")
print("should be under an hour old. 3.5h means publications are missing, not that")
print("ERCOT was late. This is the causal chain a raw file count cannot show.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Five publications are absent on 2026-03-06. One day later:

operating_date  hour_ending  worst_age_hours  zone_hours
    2026-03-07           13              2.5           8
    2026-03-07           14              3.5           8

NP3-565 publishes hourly, so the newest vintage publishable by a cutoff
should be under an hour old. 3.5h means publications are missing, not that
ERCOT was late. This is the causal chain a raw file count cannot show.


## What breaks

| Failure | How it presents | Why it is dangerous |
| --- | --- | --- |
| **Subscription key without bearer token** | `401` on every Public API endpoint | Looks like an auth typo. It is two credentials, not one. |
| **Rate limit** | `429`, or silent truncation | ERCOT documents 30 req/min. A fixed `sleep` controls the *gap*, not the *count* — `RateLimiter` counts. |
| **MIS filename read as UTC** | nothing at all | Shifts every cutoff by 5–6 hours, in the direction that admits late forecasts. Filenames are **Central**. |
| **Listing HTML changes shape** | `parse_listing` returns fewer rows | Silent under-fetch. Coverage catches it a day later; nothing catches it immediately. |
| **Partial or truncated zip** | parse error downstream | Content hashing detects it only if the hash is checked on read, not just on write. |
| **Poller down > 7 days** | **permanent data loss** | The MIS listing has already rotated. The archive may not hold the gap. |

That last row is the one to design around.

## In production

**Cadence.** NP3-565 posts hourly at `HH:30`; NP6-345 posts daily, early morning
Central, for the prior operating day. Do **not** schedule a cron at `HH:31` —
one missed run is one permanently lost vintage. Poll every 10–15 minutes and let
content hashing make re-fetches free. Four chances per publication, not one.

**Landing zone.** Object store, write-once, keyed by content hash, with a
manifest row per file recording `(report, original_filename, publication_ts,
sha256, bytes, fetched_at)`. `fetched_at` is not `publication_ts` and both are
worth keeping — the difference between them is the only direct evidence of your
own retrieval lag.

**Retention is the real risk.** With a ~7-day source window, the poller *is* the
archive for recent data. Alert on **poller staleness**, not poller failure: "no
new NP3-565 vintage in 90 minutes" is the page. A job that fails loudly gets
noticed; a job that succeeds while fetching nothing does not.

**Secrets.** Both credentials in a secret manager with rotation, not `.env`.
`.env` is right for a take-home and wrong for a deployment.

**Backfill** stays a separate, bounded, resumable job — not part of the hourly
path. It is idempotent here, which is what makes "just run it again" a safe
instruction rather than a hopeful one.

**Monitoring.** Emit `held/expected` per report per closed day. Alert when a
*closed* day is short. Do not alert on the current day — it is short by
construction until it ends.

## Runbook

| Symptom | First check | Action |
| --- | --- | --- |
| Coverage short on a closed day | Re-walk the archive listing for that day | Present in listing → our retrieval failed, re-run backfill. Absent → ERCOT gap, record it and let the readiness gate decide. |
| `401` from the API | Is a bearer token present, or only the subscription key? | Mint the token from the ercot.com account login. |
| `429` | Configured requests/minute | Lower it; confirm the limiter counts requests rather than sleeping a fixed interval. |
| No new vintage in 90 min | Is ERCOT publishing at all? | ERCOT down → wait, record the gap. Us down → restart and backfill the window **before the 7-day listing rotates**. |
| Parse failures spike | `schema_fingerprint` on recent files | Header changed → rows quarantine as `QUARANTINED_SCHEMA_ERROR` and the gate blocks. Fix the map; never map by position. |

## Where this lives

`src/forecast_spine/ercot.py` (public MIS) · `ercot_api.py` (archive, rate limiting)
· `coverage.py` · `scripts/retrieval_report.py`